In [7]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import numpy as np
from xgboost import XGBRegressor
from xrfm import xRFM
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
from sk_xRFM import SklearnXRFMRegressor


In [4]:
train.sample(2)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
178,179,20,RL,63.0,17423,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,7,2009,New,Partial,501837
1223,1224,20,RL,89.0,10680,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,10,2006,WD,Normal,137900


In [2]:
train = pd.read_csv('datasets/house_prices/train.csv')
test = pd.read_csv('datasets/house_prices/test.csv')

In [6]:
[col for col in train.columns if col not in test.columns]

['SalePrice']

In [3]:
cat_cols = [col for col in train.columns if train[col].dtype == 'O']
cat_cols.append('MSSubClass')
num_cols = [col for col in train.columns if col not in cat_cols and col != 'SalePrice']

In [17]:
na_cols = [col for col in train.columns if train[col].isna().sum()]
na_num_cols = []
for col in na_cols:
    if col not in cat_cols:
        na_num_cols.append(col)

## Preprocessing data

In [4]:
target = 'SalePrice'
X = train.drop(columns=target)
y = train[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ]
)

## Hyperparameter Tuning / Training

XGBoost

In [48]:
from hp_script import *
from xgboost import XGBRegressor

def model_builder_xgb(params):
    return XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        tree_method="hist",
        early_stopping_rounds=25,
        **params
    )

def fit_fn_xgb(model, X_tr, y_tr, X_val, y_val):
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
    return model

def predict_fn_default(model, X):
    return model.predict(X)

xgb_space = {
    "n_estimators": {"type": "int", "low": 200, "high": 1000},
    "max_depth": {"type": "int", "low": 3, "high": 8},
    "learning_rate": {"type": "float", "low": 0.01, "high": 0.2, "log": True},
    "subsample": {"type": "float", "low": 0.6, "high": 1.0},
    "colsample_bytree": {"type": "float", "low": 0.6, "high": 1.0},
    "min_child_weight": {"type": "int", "low": 1, "high": 10},
    "reg_alpha": {"type": "float", "low": 1e-8, "high": 2.0, "log": True},
    "reg_lambda": {"type": "float", "low": 1e-6, "high": 20.0, "log": True},
}

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


results = bayes_tune_model(
    X_train=X_train_processed,
    y_train=y_train,
    X_test=X_test_processed,
    y_test=y_test,
    model_builder=model_builder_xgb,
    fit_fn=fit_fn_xgb,
    predict_fn=predict_fn_default,
    param_space=xgb_space,
    n_trials=20,
    n_splits=5,
)

[I 2026-04-21 23:14:47,730] A new study created in memory with name: no-name-1de5e297-a694-4e3b-9100-ca843e8fbaac


[0]	validation_0-rmse:76846.38307
[1]	validation_0-rmse:72769.49631
[2]	validation_0-rmse:69271.02600
[3]	validation_0-rmse:65844.90340
[4]	validation_0-rmse:62574.80875
[5]	validation_0-rmse:60058.82520
[6]	validation_0-rmse:57391.69984
[7]	validation_0-rmse:54957.13314
[8]	validation_0-rmse:52637.45337
[9]	validation_0-rmse:51170.26022
[10]	validation_0-rmse:49546.50743
[11]	validation_0-rmse:47845.50213
[12]	validation_0-rmse:46364.28545
[13]	validation_0-rmse:44964.63620
[14]	validation_0-rmse:43491.97166
[15]	validation_0-rmse:42109.97254
[16]	validation_0-rmse:40952.76124
[17]	validation_0-rmse:39860.39747
[18]	validation_0-rmse:38950.56710
[19]	validation_0-rmse:38365.87266
[20]	validation_0-rmse:37891.73289
[21]	validation_0-rmse:37305.68945
[22]	validation_0-rmse:37047.86747
[23]	validation_0-rmse:36727.77705
[24]	validation_0-rmse:36025.56739
[25]	validation_0-rmse:35436.91641
[26]	validation_0-rmse:35255.65126
[27]	validation_0-rmse:34834.44911
[28]	validation_0-rmse:34389.4

[I 2026-04-21 23:14:52,654] Trial 0 finished with value: 28853.302893615357 and parameters: {'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.08960785365368121, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'reg_alpha': 3.0349658373387986e-08, 'reg_lambda': 2.1085214421249816}. Best is trial 0 with value: 28853.302893615357.


[0]	validation_0-rmse:81058.93113
[1]	validation_0-rmse:80428.94743
[2]	validation_0-rmse:79765.49896
[3]	validation_0-rmse:79203.73022
[4]	validation_0-rmse:78572.05780
[5]	validation_0-rmse:77951.80744
[6]	validation_0-rmse:77364.91289
[7]	validation_0-rmse:76811.70288
[8]	validation_0-rmse:76170.43192
[9]	validation_0-rmse:75532.78567
[10]	validation_0-rmse:74944.14946
[11]	validation_0-rmse:74445.59064
[12]	validation_0-rmse:73921.16191
[13]	validation_0-rmse:73420.21686
[14]	validation_0-rmse:72854.61243
[15]	validation_0-rmse:72310.86179
[16]	validation_0-rmse:71739.87429
[17]	validation_0-rmse:71184.23569
[18]	validation_0-rmse:70625.37303
[19]	validation_0-rmse:70195.71481
[20]	validation_0-rmse:69689.54657
[21]	validation_0-rmse:69257.26654
[22]	validation_0-rmse:68676.71580
[23]	validation_0-rmse:68222.48912
[24]	validation_0-rmse:67770.97958
[25]	validation_0-rmse:67240.14683
[26]	validation_0-rmse:66715.96820
[27]	validation_0-rmse:66298.29843
[28]	validation_0-rmse:65917.2

[I 2026-04-21 23:15:15,988] Trial 1 finished with value: 31048.85251170417 and parameters: {'n_estimators': 681, 'max_depth': 7, 'learning_rate': 0.010636066512540286, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_alpha': 3.230981763448075e-07, 'reg_lambda': 2.182940145099507e-05}. Best is trial 0 with value: 28853.302893615357.


...

[0]	validation_0-rmse:80736.32042
[1]	validation_0-rmse:79808.77342
[2]	validation_0-rmse:78906.69387
[3]	validation_0-rmse:78062.58883
[4]	validation_0-rmse:77285.85413
[5]	validation_0-rmse:76381.09895
[6]	validation_0-rmse:75580.43651
[7]	validation_0-rmse:74756.40142
[8]	validation_0-rmse:74010.96831
[9]	validation_0-rmse:73286.80142
[10]	validation_0-rmse:72507.79370
[11]	validation_0-rmse:71769.27335
[12]	validation_0-rmse:70977.29094
[13]	validation_0-rmse:70305.91252
[14]	validation_0-rmse:69551.55572
[15]	validation_0-rmse:68834.00508
[16]	validation_0-rmse:68074.56458
[17]	validation_0-rmse:67319.85116
[18]	validation_0-rmse:66749.26796
[19]	validation_0-rmse:66000.28652
[20]	validation_0-rmse:65375.32390
[21]	validation_0-rmse:64762.46978
[22]	validation_0-rmse:64158.83217
[23]	validation_0-rmse:63584.81349
[24]	validation_0-rmse:62947.56106
[25]	validation_0-rmse:62284.64135
[26]	validation_0-rmse:61674.76626
[27]	validation_0-rmse:60994.27422
[28]	validation_0-rmse:60370.8

[I 2026-04-21 23:17:46,508] Trial 18 finished with value: 28076.598344767932 and parameters: {'n_estimators': 870, 'max_depth': 3, 'learning_rate': 0.018303686676460783, 'subsample': 0.752459192919585, 'colsample_bytree': 0.6848969420823312, 'min_child_weight': 6, 'reg_alpha': 0.0005783802810457772, 'reg_lambda': 0.021680994467772063}. Best is trial 14 with value: 27447.37802992569.


[0]	validation_0-rmse:79978.48635
[1]	validation_0-rmse:78177.15858
[2]	validation_0-rmse:76797.49488
[3]	validation_0-rmse:75139.13193
[4]	validation_0-rmse:73586.40469
[5]	validation_0-rmse:72005.38488
[6]	validation_0-rmse:70564.36860
[7]	validation_0-rmse:69220.25195
[8]	validation_0-rmse:67993.19456
[9]	validation_0-rmse:66774.49394
[10]	validation_0-rmse:65444.23680
[11]	validation_0-rmse:64285.02304
[12]	validation_0-rmse:62988.36801
[13]	validation_0-rmse:61894.66410
[14]	validation_0-rmse:60856.00373
[15]	validation_0-rmse:59815.20975
[16]	validation_0-rmse:58671.90915
[17]	validation_0-rmse:57496.02903
[18]	validation_0-rmse:56481.62451
[19]	validation_0-rmse:55428.26734
[20]	validation_0-rmse:54693.36729
[21]	validation_0-rmse:53737.98243
[22]	validation_0-rmse:52980.10929
[23]	validation_0-rmse:51980.41237
[24]	validation_0-rmse:51169.81205
[25]	validation_0-rmse:50262.94866
[26]	validation_0-rmse:49646.82982
[27]	validation_0-rmse:49024.84962
[28]	validation_0-rmse:48236.7

[I 2026-04-21 23:17:51,074] Trial 19 finished with value: 28280.47108596132 and parameters: {'n_estimators': 693, 'max_depth': 5, 'learning_rate': 0.033056582966030985, 'subsample': 0.6570125731277717, 'colsample_bytree': 0.8186673537680241, 'min_child_weight': 5, 'reg_alpha': 9.517888040408765e-05, 'reg_lambda': 0.9029589573491701}. Best is trial 14 with value: 27447.37802992569.


[0]	validation_0-rmse:82108.54240
[1]	validation_0-rmse:80513.05409
[2]	validation_0-rmse:79189.26784
[3]	validation_0-rmse:77843.10749
[4]	validation_0-rmse:76494.38897
[5]	validation_0-rmse:75085.50867
[6]	validation_0-rmse:73802.70212
[7]	validation_0-rmse:72428.50486
[8]	validation_0-rmse:71126.50241
[9]	validation_0-rmse:69942.78879
[10]	validation_0-rmse:68768.72977
[11]	validation_0-rmse:67657.46837
[12]	validation_0-rmse:66488.06965
[13]	validation_0-rmse:65371.64013
[14]	validation_0-rmse:64216.40226
[15]	validation_0-rmse:63224.81546
[16]	validation_0-rmse:62253.25063
[17]	validation_0-rmse:61299.40814
[18]	validation_0-rmse:60444.12363
[19]	validation_0-rmse:59455.33718
[20]	validation_0-rmse:58567.34675
[21]	validation_0-rmse:57687.67647
[22]	validation_0-rmse:56822.56813
[23]	validation_0-rmse:56064.24138
[24]	validation_0-rmse:55243.32249
[25]	validation_0-rmse:54394.16515
[26]	validation_0-rmse:53613.03340
[27]	validation_0-rmse:52796.71845
[28]	validation_0-rmse:51985.5

xRFM

In [42]:
from hp_script import *
import numpy as np
from xrfm.xrfm import xRFM

def model_builder_xrfm(params):
    # ---- leaf params ----
    bandwidth = params.get("bandwidth", 3.0)
    reg = params.get("reg", 1e-3)
    iters = params.get("iters", 1)
    diag = params.get("diag", True)
    M_batch_size = params.get("M_batch_size", 8)

    # ---- tree params ----
    max_leaf_size = params.get("max_leaf_size", 1000)
    n_tree_iters = params.get("n_tree_iters", 0)

    rfm_params = {
        "model": {
            "kernel": "l2",
            "bandwidth": bandwidth,
            "exponent": 1.0,
            "diag": diag,
            "bandwidth_mode": "constant",
        },
        "fit": {
            "reg": reg,
            "iters": iters,
            "M_batch_size": M_batch_size,
            "verbose": False,
            "early_stop_rfm": True,
        },
    }

    return xRFM(
        rfm_params=rfm_params,
        max_leaf_size=max_leaf_size,
        n_tree_iters=n_tree_iters,
        n_trees=1,
        split_method="top_vector_agop_on_subset",
        tuning_metric='mse',
        random_state=42,
        verbose=False,
    )

def fit_fn_xrfm(model, X_tr, y_tr, X_val, y_val):
    X_tr = np.asarray(X_tr, dtype=np.float32)
    y_tr = np.asarray(y_tr, dtype=np.float32).reshape(-1, 1)
    X_val = np.asarray(X_val, dtype=np.float32)
    y_val = np.asarray(y_val, dtype=np.float32).reshape(-1, 1)

    # 🔥 duplicate target column to satisfy xRFM bug
    y_tr = np.concatenate([y_tr, y_tr], axis=1)
    y_val = np.concatenate([y_val, y_val], axis=1)

    model.fit(X_tr, y_tr, X_val, y_val)
    return model
    
def predict_fn_xrfm(model, X):
    X = np.asarray(X, dtype=np.float32)
    pred = np.asarray(model.predict(X))

    # if model returns duplicated 2-column regression output, take one column
    if pred.ndim == 2:
        return pred[:, 0]

    return pred.reshape(-1)
    
xrfm_space = {
    # leaf RFM params
    "bandwidth": {"type": "float", "low": 0.5, "high": 10.0, "log": True},
    "reg": {"type": "float", "low": 1e-5, "high": 1e-2, "log": True},
    "iters": {"type": "int", "low": 1, "high": 10},
    "diag": {"type": "categorical", "choices": [True, False]},
    "M_batch_size": {"type": "int", "low": 5, "high": 100},

    # xRFM tree params
    "max_leaf_size": {"type": "int", "low": 100, "high": 2000},
    "n_tree_iters": {"type": "int", "low": 0, "high": 10},
}

X_train_processed = preprocessor.fit_transform(X_train).astype(np.float32)
X_test_processed = preprocessor.transform(X_test).astype(np.float32)

results = bayes_tune_model(
    X_train=X_train_processed,
    y_train=np.asarray(y_train, dtype=np.float32),
    X_test=X_test_processed,
    y_test=np.asarray(y_test, dtype=np.float32),
    model_builder=model_builder_xrfm,
    fit_fn=fit_fn_xrfm,
    predict_fn=predict_fn_xrfm,
    param_space=xrfm_space,
    n_trials=20,   # start small; xRFM is expensive
    n_splits=5,    # safer than 5 for this model
)

[I 2026-04-21 22:52:51,582] A new study created in memory with name: no-name-6bc1aa64-b90e-48d7-835a-3b2b323d708a


None
Fitting xRFM with 1 trees and 9 iterations per tree


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Using top_vector_agop_on_subset split method
Getting AGOP on subset
X_train torch.Size([832, 314]) y_train torch.Size([832, 2]) X_val torch.Size([44, 314]) y_val torch.Size([44, 2])
Fitting RFM with ntrain: 832, d: 314, and nval: 44
Using cheap batch size
Optimal M batch size: 832
Using top_vector_agop_on_subset split method
Getting AGOP on subset
X_train torch.Size([499, 314]) y_train torch.Size([499, 2]) X_val torch.Size([27, 314]) y_val torch.Size([27, 2])
Fitting RFM with ntrain: 499, d: 314, and nval: 27
Using cheap batch size
Optimal M batch size: 499
Using top_vector_agop_on_subset split method
Getting AGOP on subset
X_train torch.Size([300, 314]) y_train torch.Size([300, 2]) X_val torch.Size([16, 314]) y_val torch.Size([16, 2])
Fitting RFM with ntrain: 300, d: 314, and nval: 16
Using cheap batch size
Optimal M batch size: 300
Refilling validation set, because at least one split has been made.
Fitting RFM with ntrain: 152, d: 314, and nval: 54
Time taken for round 0: 0.004999160

Using top_vector_agop_on_subset split method
Getting AGOP on subset
X_train torch.Size([832, 314]) y_train torch.Size([832, 2]) X_val torch.Size([44, 314]) y_val torch.Size([44, 2])
Fitting RFM with ntrain: 832, d: 314, and nval: 44
Using cheap batch size
Optimal M batch size: 832
Using top_vector_agop_on_subset split method
Getting AGOP on subset
X_train torch.Size([499, 314]) y_train torch.Size([499, 2]) X_val torch.Size([27, 314]) y_val torch.Size([27, 2])
Fitting RFM with ntrain: 499, d: 314, and nval: 27
Using cheap batch size
Optimal M batch size: 499
Using top_vector_agop_on_subset split method
Getting AGOP on subset
X_train torch.Size([300, 314]) y_train torch.Size([300, 2]) X_val torch.Size([16, 314]) y_val torch.Size([16, 2])
Fitting RFM with ntrain: 300, d: 314, and nval: 16
Using cheap batch size
Optimal M batch size: 300
Refilling validation set, because at least one split has been made.
Fitting RFM with ntrain: 152, d: 314, and nval: 78
Time taken for round 0: 0.012007951

...

Using SVD
Time taken for round 9: 0.10426688194274902 seconds
Using hard routing for tree prediction
Using top_vector_agop_on_subset split method
Getting AGOP on subset
X_train torch.Size([1040, 314]) y_train torch.Size([1040, 2]) X_val torch.Size([55, 314]) y_val torch.Size([55, 2])
Fitting RFM with ntrain: 1040, d: 314, and nval: 55
Using cheap batch size
Optimal M batch size: 1040
Refilling validation set, because at least one split has been made.
Fitting RFM with ntrain: 526, d: 314, and nval: 327
Using SVD
Time taken for round 0: 0.10080480575561523 seconds
Using SVD
Time taken for round 1: 0.08032488822937012 seconds
Using SVD
Time taken for round 2: 0.09208321571350098 seconds
Using SVD
Time taken for round 3: 0.07721471786499023 seconds
Early stopping at iteration 4
Refilling validation set, because at least one split has been made.
Fitting RFM with ntrain: 526, d: 314, and nval: 300
Using SVD
Time taken for round 0: 0.1252756118774414 seconds
Using SVD
Time taken for round 1: 

Building trees: 100%|██████████| 1/1 [00:16<00:00, 16.75s/it]


Time taken for round 5: 0.11629605293273926 seconds
Early stopping at iteration 6
Using hard routing for tree prediction
==========================Tree iteration results==========================
Validation scores over tree iterations: [883220032.0, 720405888.0, 679509696.0, 859088448.0, 668673984.0, 683300992.0, 819073216.0, 686102080.0, 745523072.0, 633028352.0, 586425216.0]
Best validation score: 586425216.0


Tuning split temperature: 100%|██████████| 36/36 [00:00<00:00, 61.27it/s]

Selected split_temperature=0.182074901698571 based on validation mse=581353216.000000
Using soft routing for tree prediction


MLP

In [53]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [54]:
def model_builder_keras(params, input_dim=None):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for _ in range(params["num_layers"]):
        model.add(layers.Dense(params["hidden_dim"], activation=params["activation"]))
        if params["dropout"] > 0:
            model.add(layers.Dropout(params["dropout"]))

    model.add(layers.Dense(1))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=params["lr"]),
        loss="mse"
    )
    return model

def make_keras_builder(input_dim):
    def builder(params):
        return model_builder_keras(params, input_dim=input_dim)
    return builder

def fit_fn_keras(model, X_tr, y_tr, X_val, y_val):
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=25,
        restore_best_weights=True
    )

    model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=200,
        batch_size=32,
        verbose=0,
        callbacks=[early_stop]
    )
    return model

def predict_fn_keras(model, X):
    return model.predict(X, verbose=0).reshape(-1)

keras_space = {
    "hidden_dim": {"type": "int", "low": 32, "high": 256},
    "num_layers": {"type": "int", "low": 1, "high": 4},
    "dropout": {"type": "float", "low": 0.0, "high": 0.5},
    "lr": {"type": "float", "low": 1e-4, "high": 1e-2, "log": True},
    "activation": {"type": "categorical", "choices": ["relu", "tanh"]},
}

builder = make_keras_builder(input_dim=X_train_processed.shape[1])

results = bayes_tune_model(
    X_train=X_train_processed,
    y_train=y_train,
    X_test=X_test_processed,
    y_test=y_test,
    model_builder=builder,
    fit_fn=fit_fn_keras,
    predict_fn=predict_fn_keras,
    param_space=keras_space,
    n_trials=20,
    n_splits=5,
)

[I 2026-04-21 23:20:44,321] A new study created in memory with name: no-name-e613d29a-e8df-4bc6-ba38-1d471664cfe3


[I 2026-04-21 23:21:47,062] Trial 0 finished with value: 32054.739664427478 and parameters: {'hidden_dim': 116, 'num_layers': 4, 'dropout': 0.36599697090570255, 'lr': 0.0015751320499779737, 'activation': 'relu'}. Best is trial 0 with value: 32054.739664427478.
[I 2026-04-21 23:24:43,940] Trial 1 finished with value: 197078.58478751578 and parameters: {'hidden_dim': 45, 'num_layers': 4, 'dropout': 0.3005575058716044, 'lr': 0.0026070247583707684, 'activation': 'tanh'}. Best is trial 0 with value: 32054.739664427478.
[I 2026-04-21 23:27:41,632] Trial 2 finished with value: 197412.34835317367 and parameters: {'hidden_dim': 219, 'num_layers': 1, 'dropout': 0.09091248360355031, 'lr': 0.00023270677083837802, 'activation': 'tanh'}. Best is trial 0 with value: 32054.739664427478.
[I 2026-04-21 23:30:38,471] Trial 3 finished with value: 197557.37428196045 and parameters: {'hidden_dim': 129, 'num_layers': 2, 'dropout': 0.30592644736118974, 'lr': 0.00019010245319870352, 'activation': 'tanh'}. Best

In [56]:
results

{'best_params': {'hidden_dim': 134,
  'num_layers': 4,
  'dropout': 0.09983689107917987,
  'lr': 0.0010677482709481358,
  'activation': 'relu'},
 'best_cv_score': 31920.04773247498,
 'best_model': <Sequential name=sequential_100, built=True>,
 'test_rmse': np.float64(28085.383938268686),
 'test_r2': np.float64(0.8874012859294829),
 'test_predictions': array([162263.55 , 339180.7  ,  93623.11 , 176360.78 , 365428.44 ,
         75001.2  , 246131.38 , 149807.31 ,  71466.31 , 151666.83 ,
        146672.33 , 114894.06 ,  96664.78 , 220847.66 , 177804.44 ,
        133245.83 , 194604.58 , 125957.695, 122941.53 , 219865.66 ,
        158792.88 , 209658.03 , 185800.7  , 135659.66 , 210583.9  ,
        152520.86 , 204249.14 ,  99449.87 , 175034.3  , 206786.19 ,
        130771.36 , 283753.75 , 224613.81 , 112089.78 , 263198.38 ,
        150203.39 , 134708.62 , 211366.12 , 335230.53 , 109409.87 ,
        146933.42 , 232223.8  , 113107.63 , 352419.34 , 137019.47 ,
        144576.3  , 113684.56 , 142

In [58]:
with open('results/house_prices/xgboost.pkl', 'rb') as f:
    xrfm_results = pickle.load(f)
xrfm_results

{'best_params': {'n_estimators': 827,
  'max_depth': 4,
  'learning_rate': 0.026351526093227745,
  'subsample': 0.6638123593227387,
  'colsample_bytree': 0.6052561262871234,
  'min_child_weight': 8,
  'reg_alpha': 0.15144793795413,
  'reg_lambda': 0.10581486640489153},
 'best_cv_score': 27447.37802992569,
 'best_model': XGBRegressor(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.6052561262871234, device=None,
              early_stopping_rounds=25, enable_categorical=False,
              eval_metric=None, feature_types=None, feature_weights=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.026351526093227745,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=8, missing=nan, monotone_constraint

In [55]:
import pickle
with open('results/house_prices/mlp.pkl', 'wb') as f:
    pickle.dump(results, f)

In [5]:
def htuning(model, search_spaces):
    preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ]
)
    # ----------------------------
    # 2. Full pipeline
    # ----------------------------
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # ----------------------------
    # 3. K-fold CV
    # ----------------------------
    cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # ----------------------------
    # 5. Bayesian search
    # ----------------------------
    opt = BayesSearchCV(
        estimator=pipe,
        search_spaces=search_spaces,
        n_iter=32,   # a bit leaner and more efficient
        scoring='neg_root_mean_squared_error',
        cv=cv,
        n_jobs=1,
        random_state=42,
        verbose=1,
        refit=True
    )

    # ----------------------------
    # 6. Fit
    # ----------------------------
    opt.fit(X_train, y_train)

    print("Best CV RMSE:", -opt.best_score_)
    print("Best params:")
    for k, v in opt.best_params_.items():
        print(f"{k}: {v}")

    best_model = opt.best_estimator_

    # ----------------------------
    # 7. Test evaluation
    # ----------------------------
    y_pred = best_model.predict(X_test)

    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    print("\nTest RMSE:", rmse)
    print("Test R^2:", r2)

In [14]:

xrfm = SklearnXRFMRegressor(
    tuning_metric=None,
    split_method='top_vector_agop_on_subset',
    verbose=False,
    random_state=42
)

xrfm_search_spaces = {
    'model__bandwidth': Real(0.5, 10.0, prior='log-uniform'),
    'model__reg': Real(1e-4, 1e-1, prior='log-uniform'),
    'model__iters': Integer(1, 2),
    'model__diag': Categorical([True]),
    'model__M_batch_size': Categorical([4, 8, 16, 32]),

    'model__max_leaf_size': Categorical([2000]),
    'model__n_tree_iters': Integer(1, 10),
}

htuning(xrfm, xrfm_search_spaces)

Fitting 5 folds for each of 1 candidates, totalling 5 fits
None
Fitting xRFM with 1 trees and 5 iterations per tree


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 876, d: 312, and nval: 876


Building trees:   0%|          | 0/1 [00:11<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
xrfm = xRFM(
    tuning_metric='mse',
    split_method='top_vector_agop_on_subset',
    verbose=False,
    random_state=42
)
Xt = preprocessor.fit_transform(X_train)
xrfm.fit(Xt, y_train.to_numpy(), Xt, y_train.to_numpy())

None
Fitting xRFM with 1 trees and 0 iterations per tree


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 1095, d: 314, and nval: 1095
Using cheap batch size
Optimal M batch size: 1095
Time taken for round 0: 0.04700350761413574 seconds
Using cheap batch size
Optimal M batch size: 1095
Time taken for round 1: 0.05451774597167969 seconds
Using cheap batch size
Optimal M batch size: 1095
Time taken for round 2: 0.03699970245361328 seconds
Using cheap batch size
Optimal M batch size: 1095
Time taken for round 3: 0.03299903869628906 seconds
Using cheap batch size

Building trees:   0%|          | 0/1 [00:00<?, ?it/s]


Optimal M batch size: 1095
Time taken for round 4: 0.029525279998779297 seconds
Using cheap batch size
Optimal M batch size: 1095
Tree has no split, stopping training


In [27]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

xtest = preprocessor.transform(X_test).astype(np.float32)
y_pred = xrfm.predict(xtest)
y_true = y_test.to_numpy().ravel()

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("RMSE:", rmse)
print("R2:", r2)

Using hard routing for tree prediction
RMSE: 29624.18255412291
R2: 0.8747246861457825


In [24]:
xrfm = SklearnXRFMRegressor(
    tuning_metric=None,
    split_method='top_vector_agop_on_subset',
    verbose=False,
    random_state=42
)
Xt = preprocessor.fit_transform(X_train)
xrfm.fit(Xt, y_train.to_numpy())

None
Fitting xRFM with 1 trees and 0 iterations per tree


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 1095, d: 314, and nval: 1095


Building trees:   0%|          | 0/1 [00:34<?, ?it/s]


RuntimeError: [enforce fail at alloc_cpu.cpp:117] data. DefaultCPUAllocator: not enough memory: you tried to allocate 935721256000 bytes.

In [33]:
xgbr_search_spaces = {
    'model__n_estimators': Integer(200, 1200),
    'model__max_depth': Integer(3, 8),
    'model__learning_rate': Real(0.01, 0.15, prior='log-uniform'),
    'model__subsample': Real(0.65, 1.0),
    'model__colsample_bytree': Real(0.65, 1.0),
    'model__min_child_weight': Integer(1, 12),
    'model__gamma': Real(1e-8, 2.0, prior='log-uniform'),
    'model__reg_alpha': Real(1e-8, 2.0, prior='log-uniform'),
    'model__reg_lambda': Real(1e-6, 20.0, prior='log-uniform')
}
xgbr = XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1,
        tree_method='hist'   # usually faster on tabular data
    )
xrfm = xRFM(
    tuning_metric='mse',     # Tuning metric
    split_method='top_vector_agop_on_subset'  # Splitting strategy
)
xrfm_search_spaces = {
    # ----- Leaf RFM params -----
    'model__rfm_params__model__bandwidth': Real(0.5, 20.0, prior='log-uniform'),
    'model__rfm_params__fit__reg': Real(1e-5, 1e-1, prior='log-uniform'),
    'model__rfm_params__fit__iters': Integer(2, 6),
    'model__rfm_params__model__diag': Categorical([False, True]),
    'model__rfm_params__fit__M_batch_size': Categorical([512, 1024, 2048]),

    # ----- xRFM tree params -----
    'model__max_leaf_size': Integer(5000, 50000),
    'model__n_trees': Integer(1, 4),
    'model__n_tree_iters': Integer(0, 3),
}
htuning(xrfm, xrfm_search_spaces)

None
Fitting 5 folds for each of 1 candidates, totalling 5 fits


AttributeError: 'xRFM' object has no attribute 'set_params'

In [26]:
# ----------------------------
# 1. Your preprocessing
# ----------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ]
)

# ----------------------------
# 2. Full pipeline
# ----------------------------
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1,
        tree_method='hist'   # usually faster on tabular data
    ))
])

# ----------------------------
# 3. K-fold CV
# ----------------------------
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# ----------------------------
# 4. Bayesian search space
# ----------------------------
search_spaces = {

    # Focused XGBoost ranges
    'model__n_estimators': Integer(200, 1200),
    'model__max_depth': Integer(3, 8),
    'model__learning_rate': Real(0.01, 0.15, prior='log-uniform'),
    'model__subsample': Real(0.65, 1.0),
    'model__colsample_bytree': Real(0.65, 1.0),
    'model__min_child_weight': Integer(1, 12),
    'model__gamma': Real(1e-8, 2.0, prior='log-uniform'),
    'model__reg_alpha': Real(1e-8, 2.0, prior='log-uniform'),
    'model__reg_lambda': Real(1e-6, 20.0, prior='log-uniform')
}

# ----------------------------
# 5. Bayesian search
# ----------------------------
opt = BayesSearchCV(
    estimator=pipe,
    search_spaces=search_spaces,
    n_iter=32,   # a bit leaner and more efficient
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
    refit=True
)

# ----------------------------
# 6. Fit
# ----------------------------
opt.fit(X_train, y_train)

print("Best CV RMSE:", -opt.best_score_)
print("Best params:")
for k, v in opt.best_params_.items():
    print(f"{k}: {v}")

best_model = opt.best_estimator_

# ----------------------------
# 7. Test evaluation
# ----------------------------
y_pred = best_model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("\nTest RMSE:", rmse)
print("Test R^2:", r2)

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fi